[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/36.%20The%20Berth%20%26%20Quay%20Length%20Design%20Problem/P36-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 36. The Berth & Quay Length Design Problem

## Tier 7: Human–AI Partnership

### Goal

This notebook demonstrates a human-in-the-loop decision process for berth design. The AI component generates a finite set of feasible berth configurations, evaluates them against cost, capacity, waiting-cost, and flexibility objectives, identifies the Pareto frontier, and explains its recommendation. Human experts contribute domain judgment, stakeholder priorities, and review of ethical and operational considerations.

The computation is a transparent, illustrative decision-support model. It does not train a machine-learning model, run a Monte Carlo simulation, or establish empirical performance for a real port project. Its confidence value is a heuristic score, not a statistical confidence interval.

### Inputs, outputs, and scope

**Inputs** include the berth-length and construction-budget constraints, a set of stakeholder priority weights, and explicitly specified cost assumptions. The example also defines stakeholder roles and confidence thresholds.

**Outputs** include the feasible configurations considered, objective scores, the Pareto frontier, one selected recommendation, an explanation, and a human-review status.

**Scope and modeling assumptions:** berth lengths are continuous values in meters; candidate layouts are generated using four specified distribution strategies for a fixed list of berth counts. The construction budget applies to construction cost. Operational and waiting costs are annual estimates, so their sum with construction cost is reported only as a first-year cost proxy; it is not a lifecycle-cost estimate. The example's numerical inputs are scenario assumptions, not independently verified project data. Environmental justice, worker safety, and other ethical principles are not assigned invented numeric scores: they are explicit topics for human review because the scenario provides no measurements with which to quantify them.

### Partnership workflow

1. Define stakeholders, their expertise, priorities, and confidence thresholds.
2. Generate candidate berth configurations and discard candidates that violate physical or budget constraints.
3. Score feasible candidates and find the Pareto frontier.
4. Select a recommendation from that frontier using a stated weighted score.
5. Explain the trade-offs and flag whether the recommendation requires human review.

The method is deliberately finite and inspectable: candidate generation is followed by objective evaluation, Pareto filtering, and final selection. Human review complements this computation; it does not turn the numerical scores into evidence of stakeholder approval or ethical compliance.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from enum import Enum
import math
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class EthicalPrinciple(Enum):
    """Principles used to structure human review of a recommendation."""

    NON_MALEFICENCE = "non_maleficence"
    BENEFICENCE = "beneficence"
    AUTONOMY = "autonomy"
    JUSTICE = "justice"
    TRANSPARENCY = "transparency"
    PRECAUTIONARY = "precautionary"


@dataclass
class Stakeholder:
    """A human participant with domain expertise and weighted priorities."""

    id: str
    name: str
    role: str
    expertise: List[str]
    priorities: Dict[str, float]
    confidence_threshold: float


@dataclass
class BerthConfiguration:
    """A feasible or candidate berth layout and its modeled evaluations."""

    id: str
    num_berths: int
    berth_lengths: List[float]
    total_length: float
    construction_cost: float
    operational_cost: float
    waiting_cost: float
    total_cost: float
    ai_optimization_score: float = 0.0
    ai_confidence: float = 0.0
    human_feasibility_score: Optional[float] = None
    human_stakeholder_approval: Optional[float] = None
    human_ethical_compliance: Optional[float] = None
    environmental_justice: Optional[float] = None
    worker_safety_equity: Optional[float] = None
    transparency_score: Optional[float] = None
    precautionary_compliance: Optional[float] = None

### Scenario and stakeholders

The scenario specifies a maximum quay length of 4,200 m, up to 20 berths, berth lengths between 250 m and 450 m, and a construction budget of €3.2 billion. The vessel categories are retained as context; this model does not contain vessel-by-berth compatibility data, so it does not claim to test whether a particular berth can serve a particular vessel.

A stakeholder priority is a named concern with a weight. Within each stakeholder, the weights express relative emphasis and sum to one in this example. The AI evaluation uses the priorities that have a defined mapping to its modeled objectives. Priorities that lack a modeled objective do not become numerical scores; the relevant issue remains for human assessment.

In [ ]:
def create_concrete_example() -> Tuple[List[Stakeholder], Dict]:
    """Create the stakeholder team and scenario constraints."""
    stakeholders = [
        Stakeholder(
            id="PD_001",
            name="Sarah Chen",
            role="Port Development Director",
            expertise=["strategic_planning", "infrastructure_design", "financial_analysis"],
            priorities={
                "economic_efficiency": 0.30,
                "capacity_growth": 0.25,
                "risk_management": 0.20,
                "innovation": 0.15,
                "stakeholder_relations": 0.10,
            },
            confidence_threshold=0.75,
        ),
        Stakeholder(
            id="ME_001",
            name="Marco Rodriguez",
            role="Marine Engineering Specialist",
            expertise=[
                "naval_architecture",
                "structural_engineering",
                "safety_analysis",
                "environmental_compliance",
            ],
            priorities={
                "safety": 0.40,
                "technical_feasibility": 0.30,
                "environmental_protection": 0.20,
                "operational_efficiency": 0.10,
            },
            confidence_threshold=0.85,
        ),
        Stakeholder(
            id="EA_001",
            name="Lisa Thompson",
            role="Environmental Impact Analyst",
            expertise=[
                "environmental_assessment",
                "sustainability",
                "regulatory_compliance",
                "climate_impact",
            ],
            priorities={
                "environmental_protection": 0.40,
                "sustainability": 0.30,
                "regulatory_compliance": 0.20,
                "biodiversity": 0.10,
            },
            confidence_threshold=0.80,
        ),
    ]

    constraints = {
        "max_total_length": 4200,
        "max_num_berths": 20,
        "min_berth_length": 250,
        "max_berth_length": 450,
        "construction_budget": 3_200_000_000,
        "target_vessels": ["ULCV", "Post-Panamax", "Panamax", "Feeder"],
    }
    return stakeholders, constraints


stakeholders, constraints = create_concrete_example()

print("Human expert team:")
for stakeholder in stakeholders:
    print(f"  {stakeholder.name} — {stakeholder.role}")
    print(f"    Expertise: {', '.join(stakeholder.expertise)}")
    print(f"    Priority weights: {stakeholder.priorities}")
    print(f"    Confidence threshold: {stakeholder.confidence_threshold:.0%}")

print("\nScenario constraints:")
for name, value in constraints.items():
    print(f"  {name}: {value}")

### Mathematical model and decision logic

A configuration is a vector of berth lengths, $\boldsymbol{l}=(l_1,\ldots,l_n)$, where $n$ is the number of berths. Candidate configurations must satisfy the physical limits and the construction budget:

$$
n \leq n_{\max}, \qquad
l_{\min} \leq l_i \leq l_{\max} \quad (i=1,\ldots,n), \qquad
\sum_{i=1}^{n} l_i \leq L_{\max}, \qquad
C_{\mathrm{construction}}(\boldsymbol{l}) \leq B.
$$

Here, $n_{\max}$ is the maximum berth count, $l_{\min}$ and $l_{\max}$ are the allowed berth lengths, $L_{\max}$ is the total quay-length limit, and $B$ is the construction budget. The code uses the corresponding entries in `constraints`. Candidate lengths come from four specified strategies—equal, weighted, mixed, and progressive—applied to a fixed list of berth counts. This is a finite candidate-generation procedure, not an exhaustive search over every possible real-valued layout.

For a feasible configuration, the model records construction cost, annual operational cost, and annual waiting cost. The reported `total_cost` is their sum and is interpreted as a first-year cost proxy; the budget constraint applies to construction cost alone. The objective scores are:

$$
s_{\mathrm{cost}} = \max\!\left(0, 1-\frac{C_{\mathrm{total}}}{B}\right), \qquad
s_{\mathrm{capacity}} = \frac{n}{n_{\max}}, \qquad
s_{\mathrm{flexibility}} = \frac{\operatorname{Var}(l_1,\ldots,l_n)}{(l_{\max}-l_{\min})^2/4}, \qquad
s_{\mathrm{waiting}} = \frac{1}{1+C_{\mathrm{waiting}}/C_0}.
$$

$C_0$ is the model's reference annual waiting-cost scale, calculated from the assumed annual vessel count and cost per vessel. The flexibility score measures berth-length diversity; it is not a direct measure of operational performance. These definitions are implemented in `_evaluate_alternative`.

The model treats cost as an objective to minimize and capacity and flexibility as objectives to maximize. Configuration $a$ dominates configuration $b$ if $a$ is no worse on all three Pareto objectives and strictly better on at least one. The set of configurations not dominated by another candidate is the Pareto frontier. A single recommendation is then selected from that frontier using the fixed score $0.4s_{\mathrm{cost}}+0.4s_{\mathrm{capacity}}+0.2s_{\mathrm{flexibility}}$. This final score is a decision rule for choosing among non-dominated candidates; it does not remove the trade-offs represented by the frontier.

Stakeholder priorities weight the available objective scores. The resulting scores support comparison, but they do not quantify ethical principles for which the scenario supplies no measurements. Human review is therefore a separate decision step rather than an inferred numerical result.

In [ ]:
class ExplainableAISystem:
    """Generate, evaluate, and explain a finite set of berth alternatives."""

    def __init__(self) -> None:
        self.ml_models = {
            "multi_objective_optimizer": {
                "model_type": "Pareto_Front_Explorer",
                "objectives": [
                    "cost_minimization",
                    "capacity_maximization",
                    "waiting_cost_reduction",
                    "flexibility_maximization",
                ],
                "last_updated": datetime.now() - timedelta(days=7),
            },
            "ethical_compliance_checker": {
                "model_type": "Human_Review_Checklist",
                "ethical_principles": [principle.value for principle in EthicalPrinciple],
            },
        }
        self.priority_mapping = {
            "economic_efficiency": {
                "cost": 0.7, "capacity": 0.2, "flexibility": 0.1,
            },
            "capacity_growth": {
                "cost": 0.1, "capacity": 0.7, "flexibility": 0.2,
            },
            "risk_management": {
                "cost": 0.3, "capacity": 0.3, "flexibility": 0.4,
            },
            "innovation": {
                "cost": 0.2, "capacity": 0.3, "flexibility": 0.5,
            },
            "stakeholder_relations": {
                "cost": 0.4, "capacity": 0.3, "flexibility": 0.3,
            },
            "safety": {
                "cost": 0.1, "capacity": 0.4, "flexibility": 0.3,
                "waiting_time": 0.2,
            },
            "technical_feasibility": {
                "cost": 0.2, "capacity": 0.3, "flexibility": 0.3,
                "safety": 0.2,
            },
            "environmental_protection": {
                "cost": 0.1, "capacity": 0.2, "flexibility": 0.4,
                "environmental_justice": 0.3,
            },
            "regulatory_compliance": {
                "cost": 0.1, "capacity": 0.2, "flexibility": 0.3,
                "environmental_justice": 0.4,
            },
            "biodiversity": {
                "cost": 0.1, "capacity": 0.1, "flexibility": 0.4,
                "environmental_justice": 0.4,
            },
            "operational_efficiency": {
                "cost": 0.2, "capacity": 0.3, "flexibility": 0.2,
                "waiting_time": 0.3,
            },
        }

    def generate_recommendation(
        self,
        problem_constraints: Dict,
        stakeholder_priorities: Dict[str, float],
    ) -> Dict:
        """Run candidate generation, scoring, Pareto selection, and explanation."""
        start_time = time.time()
        alternatives = self._generate_alternatives(problem_constraints)
        if not alternatives:
            raise ValueError("No feasible berth configurations were generated.")

        evaluated_alternatives = [
            self._evaluate_alternative(
                alternative, stakeholder_priorities, problem_constraints
            )
            for alternative in alternatives
        ]
        best_alternative, pareto_alternatives = self._select_pareto_optimal(
            evaluated_alternatives
        )
        reasoning = self._generate_reasoning(
            best_alternative, evaluated_alternatives, pareto_alternatives,
            problem_constraints,
        )
        confidence = self._calculate_confidence(
            best_alternative, evaluated_alternatives, problem_constraints
        )

        return {
            "configuration": best_alternative,
            "reasoning": reasoning,
            "confidence": confidence,
            "alternatives_considered": len(alternatives),
            "evaluated_alternatives": evaluated_alternatives,
            "pareto_configurations": pareto_alternatives,
            "computation_time": time.time() - start_time,
        }

    def _generate_alternatives(self, constraints: Dict) -> List[BerthConfiguration]:
        """Generate layouts using the four specified distribution strategies."""
        alternatives = []
        num_berths_options = [8, 10, 12, 15, 18, 20]
        strategies = ["equal", "weighted", "mixed", "progressive"]

        for num_berths in num_berths_options:
            if num_berths > constraints["max_num_berths"]:
                continue
            for strategy in strategies:
                if strategy == "equal":
                    berth_lengths = [
                        constraints["max_total_length"] / num_berths
                    ] * num_berths
                elif strategy == "weighted":
                    weights = np.linspace(0.8, 1.2, num_berths)
                    total_weight = float(np.sum(weights))
                    berth_lengths = [
                        constraints["max_total_length"] * weight / total_weight
                        for weight in weights
                    ]
                elif strategy == "mixed":
                    small_count = max(2, num_berths // 3)
                    large_count = max(1, num_berths // 4)
                    medium_count = num_berths - small_count - large_count
                    raw_lengths = (
                        [280.0] * small_count
                        + [350.0] * medium_count
                        + [420.0] * large_count
                    )
                    scale = constraints["max_total_length"] / sum(raw_lengths)
                    berth_lengths = [length * scale for length in raw_lengths]
                else:
                    base_length = constraints["min_berth_length"]
                    raw_lengths = [
                        base_length + 15 * index for index in range(num_berths)
                    ]
                    scale = constraints["max_total_length"] / sum(raw_lengths)
                    berth_lengths = [length * scale for length in raw_lengths]

                total_length = sum(berth_lengths)
                within_length_limits = all(
                    constraints["min_berth_length"] <= length
                    <= constraints["max_berth_length"]
                    for length in berth_lengths
                )
                if total_length > constraints["max_total_length"] or not within_length_limits:
                    continue

                construction_cost = total_length * 50_000
                if construction_cost > constraints["construction_budget"]:
                    continue

                operational_cost = self._estimate_operational_cost(berth_lengths)
                waiting_cost = self._estimate_waiting_cost(berth_lengths)
                total_cost = construction_cost + operational_cost + waiting_cost
                config = BerthConfiguration(
                    id=f"AI_{len(alternatives) + 1:03d}",
                    num_berths=num_berths,
                    berth_lengths=berth_lengths,
                    total_length=total_length,
                    construction_cost=construction_cost,
                    operational_cost=operational_cost,
                    waiting_cost=waiting_cost,
                    total_cost=total_cost,
                )
                alternatives.append(config)

        return alternatives

    def _evaluate_alternative(
        self,
        config: BerthConfiguration,
        priorities: Dict[str, float],
        constraints: Dict,
    ) -> Dict:
        """Score objectives and aggregate the supported stakeholder priorities."""
        budget = constraints["construction_budget"]
        cost_score = max(0.0, 1.0 - config.total_cost / budget)
        capacity_score = config.num_berths / constraints["max_num_berths"]

        length_range = (
            constraints["max_berth_length"] - constraints["min_berth_length"]
        )
        max_variance = length_range ** 2 / 4
        length_variance = float(np.var(config.berth_lengths))
        flexibility_score = min(1.0, length_variance / max_variance)

        reference_waiting_cost = 500 * 10_000
        waiting_score = 1.0 / (1.0 + config.waiting_cost / reference_waiting_cost)
        objective_scores = {
            "cost": cost_score,
            "capacity": capacity_score,
            "flexibility": flexibility_score,
            "waiting_time": waiting_score,
        }

        priority_scores = {}
        for priority_name, priority_weight in priorities.items():
            mapping = self.priority_mapping.get(priority_name)
            if mapping is None or priority_weight <= 0:
                continue
            available_weights = {
                objective: weight
                for objective, weight in mapping.items()
                if objective in objective_scores
            }
            weight_total = sum(available_weights.values())
            if weight_total > 0:
                priority_scores[priority_name] = sum(
                    weight * objective_scores[objective]
                    for objective, weight in available_weights.items()
                ) / weight_total

        supported_weight = sum(
            priorities[name] for name in priority_scores
        )
        total_score = (
            sum(priorities[name] * score for name, score in priority_scores.items())
            / supported_weight
            if supported_weight > 0
            else float(np.mean(list(objective_scores.values())))
        )
        config.ai_optimization_score = total_score

        return {
            "configuration": config,
            **objective_scores,
            "total_score": total_score,
            "priority_scores": priority_scores,
        }

    def _select_pareto_optimal(
        self, evaluated: List[Dict]
    ) -> Tuple[BerthConfiguration, List[BerthConfiguration]]:
        """Find non-dominated candidates and select one using the stated rule."""
        if not evaluated:
            raise ValueError("Pareto selection requires at least one candidate.")

        pareto_indices = []
        for i, candidate in enumerate(evaluated):
            candidate_cost = candidate["configuration"].total_cost
            is_dominated = False
            for j, other in enumerate(evaluated):
                if i == j:
                    continue
                other_cost = other["configuration"].total_cost
                no_worse = (
                    (other_cost < candidate_cost or math.isclose(
                        other_cost, candidate_cost, rel_tol=1e-12, abs_tol=1e-9
                    ))
                    and other["capacity"] >= candidate["capacity"] - 1e-12
                    and other["flexibility"] >= candidate["flexibility"] - 1e-12
                )
                strictly_better = (
                    other_cost < candidate_cost and not math.isclose(
                        other_cost, candidate_cost, rel_tol=1e-12, abs_tol=1e-9
                    )
                ) or (
                    other["capacity"] > candidate["capacity"]
                    and not math.isclose(
                        other["capacity"], candidate["capacity"],
                        rel_tol=1e-12, abs_tol=1e-12,
                    )
                ) or (
                    other["flexibility"] > candidate["flexibility"]
                    and not math.isclose(
                        other["flexibility"], candidate["flexibility"],
                        rel_tol=1e-12, abs_tol=1e-12,
                    )
                )
                if no_worse and strictly_better:
                    is_dominated = True
                    break
            if not is_dominated:
                pareto_indices.append(i)

        if not pareto_indices:
            raise RuntimeError("The evaluated alternatives have no Pareto candidate.")

        def selection_score(index: int) -> float:
            item = evaluated[index]
            return (
                0.4 * item["cost"]
                + 0.4 * item["capacity"]
                + 0.2 * item["flexibility"]
            )

        best_index = max(pareto_indices, key=selection_score)
        pareto_configurations = [
            evaluated[index]["configuration"] for index in pareto_indices
        ]
        return evaluated[best_index]["configuration"], pareto_configurations

    def _generate_reasoning(
        self,
        best_config: BerthConfiguration,
        all_evaluated: List[Dict],
        pareto_configurations: List[BerthConfiguration],
        constraints: Dict,
    ) -> List[str]:
        """Summarize the selected configuration and its modeled trade-offs."""
        reasoning = [
            f"Selected {best_config.id} from the Pareto frontier using the stated weighted score.",
            f"Layout: {best_config.num_berths} berths; lengths in meters: "
            f"{[round(length, 1) for length in best_config.berth_lengths]}.",
            f"Total quay length: {best_config.total_length:.1f} m "
            f"(maximum {constraints['max_total_length']} m).",
            f"Construction cost: €{best_config.construction_cost / 1e6:.2f} million; "
            f"first-year cost proxy: €{best_config.total_cost / 1e6:.2f} million.",
            f"Pareto frontier size: {len(pareto_configurations)} of "
            f"{len(all_evaluated)} feasible alternatives.",
        ]

        cost_ranking = sorted(
            all_evaluated,
            key=lambda item: item["configuration"].total_cost,
        )
        capacity_ranking = sorted(
            all_evaluated,
            key=lambda item: item["configuration"].num_berths,
            reverse=True,
        )
        reasoning.append(
            f"Across the candidate set, the first-year cost proxy ranges from "
            f"€{cost_ranking[0]['configuration'].total_cost / 1e6:.2f} million "
            f"to €{cost_ranking[-1]['configuration'].total_cost / 1e6:.2f} million."
        )
        reasoning.append(
            f"The candidate set spans {capacity_ranking[-1]['configuration'].num_berths} "
            f"to {capacity_ranking[0]['configuration'].num_berths} berths."
        )
        reasoning.append(
            "The flexibility score represents berth-length diversity, not a validated "
            "measure of vessel compatibility or operational performance."
        )
        reasoning.append(
            "Environmental justice, worker safety, and stakeholder approval require "
            "human assessment; this model does not infer or score them."
        )
        return reasoning

    def _calculate_confidence(
        self,
        best_config: BerthConfiguration,
        all_evaluated: List[Dict],
        constraints: Dict,
    ) -> float:
        """Calculate a bounded heuristic score, not a statistical interval."""
        base_confidence = 0.85
        if len(all_evaluated) > 1:
            score_variance = float(
                np.var([item["total_score"] for item in all_evaluated])
            )
            consensus_penalty = min(1.0, score_variance / 0.25)
            confidence_consensus = base_confidence * (1.0 - consensus_penalty)
        else:
            confidence_consensus = base_confidence

        length_utilization = (
            best_config.total_length / constraints["max_total_length"]
        )
        length_penalty = max(0.0, length_utilization - 0.9) * 0.1
        confidence = confidence_consensus * (1.0 - length_penalty)
        return max(0.5, min(0.95, confidence))

    def _estimate_operational_cost(self, berth_lengths: List[float]) -> float:
        """Estimate annual operational cost using the scenario's simple rule."""
        operational_cost = sum(berth_lengths) * 5_000
        num_berths = len(berth_lengths)
        if num_berths > 10:
            operational_cost *= 1.0 + (num_berths - 10) * 0.02
        return operational_cost

    def _estimate_waiting_cost(self, berth_lengths: List[float]) -> float:
        """Estimate annual waiting cost from the stated capacity proxy."""
        total_capacity = sum(berth_lengths) / 300
        utilization_factor = min(1.0, total_capacity / 1000)
        if utilization_factor > 0.8:
            waiting_multiplier = 1.0 + 5.0 * (utilization_factor - 0.8) ** 2
        else:
            waiting_multiplier = 1.0
        return 500 * 10_000 * waiting_multiplier


def aggregate_stakeholder_priorities(
    stakeholder_list: List[Stakeholder],
) -> Dict[str, float]:
    """Average each named priority across stakeholders who assign it a weight."""
    totals: Dict[str, float] = {}
    counts: Dict[str, int] = {}
    for stakeholder in stakeholder_list:
        for priority_name, weight in stakeholder.priorities.items():
            totals[priority_name] = totals.get(priority_name, 0.0) + weight
            counts[priority_name] = counts.get(priority_name, 0) + 1
    return {
        name: totals[name] / counts[name]
        for name in totals
    }


def assess_human_review(
    recommendation: Dict,
    stakeholder_list: List[Stakeholder],
) -> Dict:
    """Flag confidence-threshold escalation and list ethical review topics."""
    confidence = recommendation["confidence"]
    reviews = [
        {
            "stakeholder": stakeholder.name,
            "role": stakeholder.role,
            "threshold": stakeholder.confidence_threshold,
            "review_required": confidence < stakeholder.confidence_threshold,
        }
        for stakeholder in stakeholder_list
    ]
    return {
        "stakeholder_reviews": reviews,
        "human_review_required": any(
            review["review_required"] for review in reviews
        ),
        "ethical_review_topics": [principle.value for principle in EthicalPrinciple],
    }

### Run the decision-support workflow

The stakeholder priority aggregation gives each stakeholder equal representation when averaging a priority that multiple people share; it does not imply that stakeholders have equal authority in a real port decision. The resulting priority weights are passed explicitly to the evaluator. The human-review step compares the heuristic recommendation score with each stated confidence threshold and lists ethical topics for review. A review flag is not an approval, an override, or evidence that a stakeholder has examined the recommendation.

In [ ]:
ai_system = ExplainableAISystem()
combined_priorities = aggregate_stakeholder_priorities(stakeholders)
recommendation = ai_system.generate_recommendation(
    constraints,
    combined_priorities,
)
human_review = assess_human_review(recommendation, stakeholders)

selected = recommendation["configuration"]
assert selected.num_berths <= constraints["max_num_berths"]
assert selected.total_length <= constraints["max_total_length"] + 1e-9
assert selected.construction_cost <= constraints["construction_budget"]
assert all(
    constraints["min_berth_length"] - 1e-9
    <= length
    <= constraints["max_berth_length"] + 1e-9
    for length in selected.berth_lengths
)

print("AI recommendation")
print(f"  Configuration: {selected.id}")
print(f"  Berths: {selected.num_berths}")
print(f"  Total quay length: {selected.total_length:.1f} m")
print(f"  Construction cost: €{selected.construction_cost / 1e6:.2f} million")
print(f"  First-year cost proxy: €{selected.total_cost / 1e6:.2f} million")
print(f"  Heuristic confidence score: {recommendation['confidence']:.1%}")
print(f"  Feasible alternatives considered: {recommendation['alternatives_considered']}")
print(f"  Pareto-frontier alternatives: {len(recommendation['pareto_configurations'])}")
print(f"  Computation time: {recommendation['computation_time']:.4f} seconds")

print("\nExplanation")
for number, reason in enumerate(recommendation["reasoning"], start=1):
    print(f"  {number}. {reason}")

print("\nHuman-review status")
print(f"  Any confidence-threshold review required: {human_review['human_review_required']}")
for review in human_review["stakeholder_reviews"]:
    status = "Review required" if review["review_required"] else "Threshold met"
    print(
        f"  {review['name'] if 'name' in review else review['stakeholder']} "
        f"({review['role']}): {status}; threshold {review['threshold']:.0%}"
    )
print("  Ethical review topics: " + ", ".join(human_review["ethical_review_topics"]))

### Inspect candidate scores

The table makes the decision auditable: it reports each generated candidate's berth count, cost components, objective scores, and stakeholder-weighted score. Pareto status indicates whether another candidate is at least as good on every Pareto objective and strictly better on one. A non-dominated candidate is not necessarily the preferred real-world design; it is a candidate for human deliberation under the objectives represented here.

In [ ]:
pareto_ids = {
    config.id for config in recommendation["pareto_configurations"]
}
table_rows = []
for item in recommendation["evaluated_alternatives"]:
    config = item["configuration"]
    table_rows.append(
        {
            "Configuration": config.id,
            "Berths": config.num_berths,
            "Total length (m)": config.total_length,
            "Construction cost (€M)": config.construction_cost / 1e6,
            "Annual operational cost (€M)": config.operational_cost / 1e6,
            "Annual waiting cost (€M)": config.waiting_cost / 1e6,
            "First-year cost proxy (€M)": config.total_cost / 1e6,
            "Cost score": item["cost"],
            "Capacity score": item["capacity"],
            "Flexibility score": item["flexibility"],
            "Waiting score": item["waiting_time"],
            "Stakeholder-weighted score": item["total_score"],
            "Pareto frontier": config.id in pareto_ids,
        }
    )

candidate_table = pd.DataFrame(table_rows).sort_values(
    by=["Pareto frontier", "Stakeholder-weighted score"],
    ascending=[False, False],
)
display(candidate_table.reset_index(drop=True).round(3))

### Visualize the cost–capacity trade-off

The horizontal axis shows the modeled first-year cost proxy, while the vertical axis shows berth count as a capacity proxy. Color distinguishes Pareto-frontier candidates from other feasible candidates, and the star marks the selected recommendation. The chart shows only two dimensions: flexibility remains part of Pareto filtering and final selection even though it is not displayed on an axis.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for is_pareto, group in candidate_table.groupby("Pareto frontier"):
    label = "Pareto frontier" if is_pareto else "Other feasible candidates"
    ax.scatter(
        group["First-year cost proxy (€M)"],
        group["Berths"],
        label=label,
        alpha=0.8,
        s=65,
    )

ax.scatter(
    selected.total_cost / 1e6,
    selected.num_berths,
    marker="*",
    s=240,
    color="black",
    label="Selected recommendation",
    zorder=5,
)
ax.set_title("Feasible Berth Configurations: Cost–Capacity View")
ax.set_xlabel("First-year cost proxy (€ millions)")
ax.set_ylabel("Number of berths (capacity proxy)")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

### Interpretation, limitations, and appropriate use

This workflow combines computational screening with human judgment: the algorithm can compare candidate configurations consistently, while domain experts can examine assumptions, missing evidence, stakeholder consequences, and ethical trade-offs. The Pareto frontier preserves objective trade-offs; the fixed weighted score selects one candidate for discussion rather than proving that it is universally best.

The confidence score is a heuristic based on variation in candidate scores and use of the total-length limit. It is not a calibrated probability, confidence interval, or measure of model accuracy. The cost estimates are simplified, the candidate set is restricted to the stated generation strategies, and the model does not include vessel-arrival schedules, berth compatibility, quay geometry beyond total length, environmental-impact measurements, construction phasing, or lifecycle discounting. A real investment decision would require validated data and appropriate human, regulatory, and stakeholder review.

Use this partnership pattern when a decision has competing priorities and benefits from transparent computational comparisons, but do not treat the generated ranking as a substitute for engineering validation, ethical analysis, or stakeholder authority. Natural extensions within the same workflow include adding validated constraints and measurements to candidate evaluation, then documenting how those inputs affect the Pareto frontier and human review.